# MCP Concept Demo (All-in-One) — Teaching Notebook

This notebook demonstrates the core idea behind **MCP (Model Context Protocol)** in a **simple, classroom-friendly** way:

- A **server** exposes a set of **tools** (functions).
- A **client** calls tools using **structured arguments** (a dict/JSON-like payload).
- Tools return **grounded outputs** that an agent can use.

> ✅ This is an *educational mini-demo* that mimics the MCP pattern in **pure Python**, so students can understand the idea without setup complexity.


## 1) Learning outcomes

By the end, students can:
1. Explain what “tool calling” means (function registry + structured args).
2. Implement a tiny MCP-like server and client.
3. Add new tools safely (clear inputs/outputs).
4. Extend it to a mini “RAG-like” lookup tool.


## 2) One-cell MCP-like implementation (Server + Tools + Client)

Run the cell below. It defines:
- `MCPServer`: registers tools
- `@server.tool(name)`: decorator to expose a tool
- `MCPClient`: calls tools with a payload dict

Then it runs 3 example tool calls:
- `add(a,b)`
- `greet(name)`
- `kb_lookup(question)` (a tiny “RAG-like” lookup)


In [1]:
# ==============================
# SIMPLE MCP CONCEPT DEMO
# Server + Client in ONE CELL
# ==============================

# ----- MCP-Like Server -----
class MCPServer:
    def __init__(self):
        self.tools = {}

    def tool(self, name):
        """Decorator to register a function as a tool."""
        def decorator(func):
            self.tools[name] = func
            return func
        return decorator

    def call(self, name, args):
        """Call a registered tool with structured args (dict)."""
        if name not in self.tools:
            raise KeyError(f"Tool '{name}' not found. Available: {list(self.tools.keys())}")
        return self.tools[name](**args)


# ----- Create Server -----
server = MCPServer()


# ----- Define Tools -----
@server.tool("add")
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b


@server.tool("greet")
def greet(name: str) -> str:
    """Return a greeting"""
    return f"Hello {name}! This reply comes from a tool."


@server.tool("kb_lookup")
def kb_lookup(question: str) -> dict:
    """A tiny 'RAG-like' lookup tool returning a grounded snippet + citation."""
    kb = [
        {"key": "add/drop deadline", "answer": "Add/Drop deadline is Week 2 Friday.", "source": "Student Handbook §2.1"},
        {"key": "tuition fee", "answer": "Tuition fee is SGD 20,000 per year (example).", "source": "Fees Page §1.0"},
        {"key": "prerequisite", "answer": "A prerequisite is a module you must pass before taking another module.", "source": "Academic Rules §3.2"},
    ]
    q = question.lower()

    for item in kb:
        if item["key"] in q:
            return {"answer": item["answer"], "citation": item["source"], "confidence": 0.9}

    return {"answer": "Not found in knowledge base.", "citation": None, "confidence": 0.2}


# ----- MCP Client -----
class MCPClient:
    def __init__(self, server: MCPServer):
        self.server = server

    def call(self, tool: str, args: dict):
        print(f"\n[Client] Calling tool → {tool} with {args}")
        result = self.server.call(tool, args)
        print(f"[Server Reply] {result}")
        return result


# ----- Run Demo -----
client = MCPClient(server)

client.call("add", {"a": 5, "b": 7})
client.call("greet", {"name": "Dr Teoh"})
client.call("kb_lookup", {"question": "What is the add/drop deadline?"})



[Client] Calling tool → add with {'a': 5, 'b': 7}
[Server Reply] 12

[Client] Calling tool → greet with {'name': 'Dr Teoh'}
[Server Reply] Hello Dr Teoh! This reply comes from a tool.

[Client] Calling tool → kb_lookup with {'question': 'What is the add/drop deadline?'}
[Server Reply] {'answer': 'Add/Drop deadline is Week 2 Friday.', 'citation': 'Student Handbook §2.1', 'confidence': 0.9}


{'answer': 'Add/Drop deadline is Week 2 Friday.',
 'citation': 'Student Handbook §2.1',
 'confidence': 0.9}

## 3) Mini “Agent” that decides which tool to call

A real agent uses the user’s question to pick tools.  
Below is a **very simple rule-based router** (great for teaching).

Try different questions like:
- “Hi”
- “What is tuition fee?”
- “Explain prerequisite”
- “Calculate 123 + 456”


In [2]:
import re

def mini_agent_answer(question: str) -> str:
    q = question.strip().lower()

    # Simple routing rules (teaching demo)
    if re.search(r"\b(hi|hello|hey)\b", q):
        out = client.call("greet", {"name": "Student"})
        return out

    # math detection
    m = re.search(r"(\d+)\s*\+\s*(\d+)", q)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        out = client.call("add", {"a": a, "b": b})
        return f"Result: {out}"

    # otherwise try knowledge base lookup
    kb = client.call("kb_lookup", {"question": question})
    if kb["confidence"] >= 0.5:
        return f"{kb['answer']} (Source: {kb['citation']})"
    return "I’m not sure based on the available documents. Please rephrase or provide more info."


# Try a few questions
tests = [
    "Hi!",
    "What is 123 + 456?",
    "Explain prerequisite with a simple example.",
    "What is tuition fee?",
    "When is graduation ceremony?"  # not in KB
]

for t in tests:
    print("\nQ:", t)
    print("A:", mini_agent_answer(t))



Q: Hi!

[Client] Calling tool → greet with {'name': 'Student'}
[Server Reply] Hello Student! This reply comes from a tool.
A: Hello Student! This reply comes from a tool.

Q: What is 123 + 456?

[Client] Calling tool → add with {'a': 123, 'b': 456}
[Server Reply] 579
A: Result: 579

Q: Explain prerequisite with a simple example.

[Client] Calling tool → kb_lookup with {'question': 'Explain prerequisite with a simple example.'}
[Server Reply] {'answer': 'A prerequisite is a module you must pass before taking another module.', 'citation': 'Academic Rules §3.2', 'confidence': 0.9}
A: A prerequisite is a module you must pass before taking another module. (Source: Academic Rules §3.2)

Q: What is tuition fee?

[Client] Calling tool → kb_lookup with {'question': 'What is tuition fee?'}
[Server Reply] {'answer': 'Tuition fee is SGD 20,000 per year (example).', 'citation': 'Fees Page §1.0', 'confidence': 0.9}
A: Tuition fee is SGD 20,000 per year (example). (Source: Fees Page §1.0)

Q: When i

## 4) Teaching Activity (10–15 mins)

### Task A (Easy)
Add a new tool called `multiply(a,b)` and update the mini-agent to detect `a * b`.

### Task B (Medium)
Improve `kb_lookup` so it can match **keywords** (e.g., “deadline” matches “add/drop deadline”).

### Task C (Advanced)
Add a `policy_check(text)` tool that flags:
- missing citation
- overconfident claims (confidence < 0.5 but still answered)

Then update the mini-agent to run `policy_check` before final output.


## 5) Extension: How this maps to real MCP + LangChain

In production:
- Tools are hosted by an MCP server process (STDIO / HTTP transports)
- A LangChain agent chooses tools via structured tool calls
- RAG tools actually retrieve from embeddings/vector DB
- Policy tools enforce constraints and reduce hallucinations

This notebook gives the same mental model with minimal moving parts.
